# ARC-AGI-3 structured-memory screen
Local-only Duck comparison. No competition submission.


In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Gate 2 imports the analyzer at the control setting; each sequential trial then sets its own cap and paired seed before constructing agents.
os.environ["LOCAL_ANALYZER_MAX_OUTPUT"] = "0"
os.environ["LOCAL_ANALYZER_SEED"] = "-1"

# Gate 1 safe baseline: the prior C8 run is the comparison; validate C16 here.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c16-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "16",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Setup returns only after the production vLLM endpoint is ready.
GATE1_SERVER_READY_EPOCH = time.time()
GATE1_SERVER_STARTUP_SECONDS = GATE1_SERVER_READY_EPOCH - NOTEBOOK_START_EPOCH
print(
    f"GATE1_SERVER_READY epoch={GATE1_SERVER_READY_EPOCH} "
    f"startup_seconds={GATE1_SERVER_STARTUP_SECONDS}",
    flush=True,
)

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Exact tested memory adapter embedded for offline Kaggle execution.
from pathlib import Path as _memory_Path
import sys as _memory_sys
_memory_package = WORKING_DIR / 'arc3'
_memory_package.mkdir(parents=True, exist_ok=True)
(_memory_package / '__init__.py').write_text('')
(_memory_package / 'effects.py').write_text('"""Effect ledger: what each action actually did, per state and overall.\n\nTwo jobs. Per state, it stops the agent re-probing an action that did nothing here\nbefore -- under a squared metric a repeated no-op is pure loss. Globally, it learns\nwhich actions matter in *this* game, so exploration starts with the ones that have\npaid off rather than cycling the full action space.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\n\nActionKey = tuple[int, int | None, int | None]  # (action id, x, y) -- x/y for ACTION6\n\n\n@dataclass\nclass ActionStats:\n    attempts: int = 0\n    changes: int = 0  # times the board actually moved\n    level_advances: int = 0\n    cells_changed: int = 0\n\n    @property\n    def change_rate(self) -> float:\n        return self.changes / self.attempts if self.attempts else 0.0\n\n    @property\n    def mean_cells_changed(self) -> float:\n        return self.cells_changed / self.changes if self.changes else 0.0\n\n\n@dataclass\nclass EffectLedger:\n    """Per-game record of action outcomes."""\n\n    by_action: dict[ActionKey, ActionStats] = field(\n        default_factory=lambda: defaultdict(ActionStats)\n    )\n    # (state hash, action) -> did anything change last time we tried it here\n    by_state: dict[tuple[str, ActionKey], bool] = field(default_factory=dict)\n\n    def record(\n        self,\n        state_hash: str,\n        action: ActionKey,\n        changed_cells: int,\n        level_advanced: bool,\n    ) -> None:\n        stats = self.by_action[action]\n        stats.attempts += 1\n        if changed_cells > 0:\n            stats.changes += 1\n            stats.cells_changed += changed_cells\n        if level_advanced:\n            stats.level_advances += 1\n        self.by_state[(state_hash, action)] = changed_cells > 0\n\n    def is_known_noop(self, state_hash: str, action: ActionKey) -> bool:\n        """True if this action did nothing the last time it was tried here."""\n        return self.by_state.get((state_hash, action)) is False\n\n    def tried_here(self, state_hash: str, action: ActionKey) -> bool:\n        return (state_hash, action) in self.by_state\n\n    def attempts(self, action: ActionKey) -> int:\n        stats = self.by_action.get(action)\n        return stats.attempts if stats else 0\n\n    def exploration_rank(self, action: ActionKey) -> tuple[int, float]:\n        """Sort key for choosing among actions untried in the current state.\n\n        Breadth first, then what has worked. Ranking purely by past success\n        collapses the agent onto whichever action moved the board most often,\n        which it then repeats forever -- it stops covering the action space and\n        the run stalls. Fewest global attempts wins, with effectiveness only\n        breaking ties, so all actions keep getting exercised.\n\n        Returned for use with ``min``; lower is better.\n        """\n        return (self.attempts(action), -self.value(action))\n\n    def value(self, action: ActionKey) -> float:\n        """How useful this action has proven, ignoring novelty."""\n        stats = self.by_action.get(action)\n        if stats is None or stats.attempts == 0:\n            return 1.0  # unknown actions carry the most information\n        if stats.level_advances:\n            return 0.9\n        return stats.change_rate * 0.8\n\n    def useful_colours(self) -> set[int]:\n        """Placeholder for colour bias; filled in by the agent that owns it."""\n        return set()\n\n    def reset_level(self) -> None:\n        """Per-state facts do not survive a level; per-action tendencies do."""\n        self.by_state.clear()\n')
(_memory_package / 'memory.py').write_text('"""Structured per-game memory the model commits explicitly.\n\nDuck keeps 30 assistant turns and clears six summary fields at a level\ntransition. The defect is not the clearing: durable summaries update only from\nassistant `content`, and the model reasons then emits a tool call, so the channel\nDuck reads is frequently empty. Preserving those summaries preserves nothing.\n**The model has to commit memory explicitly**, as a typed argument beside its\naction.\n\nEvery rule below exists because it is a way to silently destroy knowledge.\n\n**All-or-nothing.** The WHOLE update is validated -- every field, to its leaves --\nbefore a single byte is written. A half-applied update leaves memory in a state\nno one designed.\n\n**Omission keeps, empty never clears.** A field not mentioned retains its value;\n`[]` means "nothing to add", not "forget". A model truncated under length\npressure must not wipe the store.\n\n**Conflict is recorded, not resolved.** Status is not a precedence ladder. A\nREFUTED claim re-proposed as a guess is blocked; a CONFIRMED claim later refuted\nbecomes CONTESTED with both evidence sets intact. An earlier version let a guess\nresurrect a disproven claim and made a wrong confirmation permanently\nuncorrectable -- opposite errors from the same mistake of ordering statuses.\n\n**Evidence must exist.** A fact citing transition 999999 of a 300-transition\nledger is an assertion wearing a citation. Pass `known_transitions` and it is\nchecked.\n\n**A winning path is not a list of actions.** Replaying one into a level it was\nnot learned in, from a state it does not fit, is worse than not having it.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom dataclasses import dataclass, field, replace\nfrom enum import Enum\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nfrom arc3.effects import ActionKey\n\n\nclass Status(str, Enum):\n    CONFIRMED = "CONFIRMED"  # checked against recorded transitions\n    ASSUMED = "ASSUMED"      # plausible, untested\n    REFUTED = "REFUTED"      # contradicted; kept so it is not re-proposed\n    CONTESTED = "CONTESTED"  # confirmed AND refuted; needs a discriminating probe\n\n\nEVIDENCE_REQUIRED = (Status.CONFIRMED, Status.REFUTED, Status.CONTESTED)\n\n\n@dataclass(frozen=True)\nclass Fact:\n    statement: str\n    status: Status = Status.ASSUMED\n    evidence: tuple[int, ...] = ()  # transition indices in this game\'s ledger\n\n    def __post_init__(self) -> None:\n        if not isinstance(self.statement, str) or not self.statement.strip():\n            raise ValueError("a fact with no statement carries no information")\n        if not isinstance(self.status, Status):\n            raise ValueError(f"status must be a Status, got {self.status!r}")\n        if any(not isinstance(i, int) or isinstance(i, bool) for i in self.evidence):\n            raise ValueError(f"evidence must be transition indices: {self.evidence!r}")\n        if self.status in EVIDENCE_REQUIRED and not self.evidence:\n            raise ValueError(\n                f"{self.status.value} without evidence: {self.statement!r}. "\n                "The store cannot tell knowledge from assertion otherwise"\n            )\n\n\n@dataclass(frozen=True)\nclass WinningPath:\n    """A sequence that worked, with everything needed to know when it applies.\n\n    Carrying bare actions across a level transition is how a \'memory\' feature\n    makes an agent worse: the board may have been resized, the objects renamed,\n    the mechanic changed. A path is replayable only from a matching level and\n    start signature, and only while each step\'s expectation holds.\n    """\n\n    level: int\n    start_signature: str  # raw board hash the path was learned from\n    actions: tuple[ActionKey, ...]\n    expected_outcomes: tuple[str, ...] = ()  # after-hash expected at each step\n    evidence: tuple[int, ...] = ()\n\n    def __post_init__(self) -> None:\n        if not self.actions:\n            raise ValueError("a winning path with no actions is not a path")\n        for action in self.actions:\n            _check_action(action)\n        if not self.start_signature:\n            raise ValueError("a path without a start signature cannot be matched")\n        if self.expected_outcomes and len(self.expected_outcomes) != len(self.actions):\n            raise ValueError("expected_outcomes must align 1:1 with actions")\n\n    def usable_from(self, level: int, signature: str) -> bool:\n        """Never replay unless the precondition matches exactly."""\n        return level == self.level and signature == self.start_signature\n\n\nDURABLE = ("action_semantics", "mechanics", "goal_evidence", "counterexamples",\n           "winning_paths")\nVOLATILE = ("current_plan", "level_coordinates", "goal_guesses", "object_identities")\nFACT_LISTS = ("mechanics", "goal_evidence", "counterexamples", "goal_guesses")\n\n\n@dataclass\nclass MemoryUpdate:\n    """What the model commits alongside its action. ``None`` means "not touching"."""\n\n    action_semantics: dict[ActionKey, list[Fact]] | None = None\n    mechanics: list[Fact] | None = None\n    goal_evidence: list[Fact] | None = None\n    counterexamples: list[Fact] | None = None\n    winning_paths: list[WinningPath] | None = None\n    current_plan: list[ActionKey] | None = None\n    level_coordinates: dict[str, Any] | None = None\n    goal_guesses: list[Fact] | None = None\n    object_identities: dict[str, Any] | None = None\n\n    def touched(self) -> list[str]:\n        return [n for n in DURABLE + VOLATILE if getattr(self, n) is not None]\n\n\ndef _check_action(action: Any) -> None:\n    if (not isinstance(action, tuple) or len(action) != 3\n            or not isinstance(action[0], int) or isinstance(action[0], bool)\n            or any(not (v is None or (isinstance(v, int) and not isinstance(v, bool)))\n                   for v in action[1:])):\n        raise ValueError(f"not an action key (id, x, y): {action!r}")\n\n\ndef _check_jsonable(name: str, value: Any) -> None:\n    try:\n        json.dumps(value, default=_json_default)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f"{name} must be JSON-serialisable: {exc}") from exc\n\n\ndef _json_default(obj: Any) -> Any:\n    if isinstance(obj, tuple):\n        return list(obj)\n    raise TypeError(f"{type(obj).__name__} is not serialisable")\n\n\ndef validate(update: MemoryUpdate, known_transitions: set[int] | None = None) -> None:\n    """Validate EVERY field to its leaves. Raises before anything is written."""\n    for name in FACT_LISTS:\n        value = getattr(update, name)\n        if value is None:\n            continue\n        if not isinstance(value, list) or any(not isinstance(f, Fact) for f in value):\n            raise ValueError(f"{name} must be a list of Fact, got {value!r}")\n        _check_evidence(name, value, known_transitions)\n\n    if update.action_semantics is not None:\n        if not isinstance(update.action_semantics, dict):\n            raise ValueError("action_semantics must be a dict keyed by action")\n        for key, facts in update.action_semantics.items():\n            _check_action(key)\n            if not isinstance(facts, list) or any(not isinstance(f, Fact) for f in facts):\n                raise ValueError(f"action_semantics[{key}] must be a list of Fact")\n            _check_evidence("action_semantics", facts, known_transitions)\n\n    if update.winning_paths is not None:\n        if not isinstance(update.winning_paths, list) or any(\n                not isinstance(p, WinningPath) for p in update.winning_paths):\n            raise ValueError("winning_paths must be a list of WinningPath")\n        if known_transitions is not None:\n            for path in update.winning_paths:\n                _check_indices("winning_paths", path.evidence, known_transitions)\n\n    if update.current_plan is not None:\n        if not isinstance(update.current_plan, list):\n            raise ValueError(f"current_plan must be a list, got {update.current_plan!r}")\n        for action in update.current_plan:\n            _check_action(action)\n\n    for name in ("level_coordinates", "object_identities"):\n        value = getattr(update, name)\n        if value is None:\n            continue\n        if not isinstance(value, dict):\n            raise ValueError(f"{name} must be a dict, got {value!r}")\n        _check_jsonable(name, value)\n\n\ndef _check_evidence(name: str, facts: Iterable[Fact],\n                    known: set[int] | None) -> None:\n    if known is None:\n        return\n    for fact in facts:\n        _check_indices(f"{name}: {fact.statement!r}", fact.evidence, known)\n\n\ndef _check_indices(label: str, indices: Iterable[int], known: set[int]) -> None:\n    missing = [i for i in indices if i not in known]\n    if missing:\n        raise ValueError(\n            f"{label} cites transitions that do not exist: {missing}. "\n            "A citation to nothing is an assertion wearing a citation"\n        )\n\n\ndef _revise(incoming: Fact, existing: Fact) -> Fact | None:\n    """Belief revision for one statement. None means the incoming fact is blocked.\n\n    Not a precedence ladder:\n\n    ```\n    existing    incoming     result\n    ASSUMED     CONFIRMED    upgrade, merge evidence\n    ASSUMED     REFUTED      refute, merge evidence\n    CONFIRMED   ASSUMED      blocked -- a guess cannot unseat a verified claim\n    CONFIRMED   REFUTED      CONTESTED, both evidence sets kept\n    REFUTED     ASSUMED      blocked -- a disproven claim is not re-proposed\n    REFUTED     CONFIRMED    CONTESTED, both evidence sets kept\n    CONTESTED   anything     stays CONTESTED, evidence accumulates\n    ```\n    """\n    merged = tuple(dict.fromkeys(existing.evidence + incoming.evidence))\n\n    if existing.status is Status.CONTESTED:\n        return replace(existing, evidence=merged)\n    if incoming.status is Status.ASSUMED and existing.status is not Status.ASSUMED:\n        return None\n    if existing.status is Status.ASSUMED:\n        return replace(incoming, evidence=merged)\n    if existing.status == incoming.status:\n        return replace(existing, evidence=merged)\n    # CONFIRMED vs REFUTED, in either direction.\n    return replace(existing, status=Status.CONTESTED, evidence=merged)\n\n\ndef _merge_facts(existing: list[Fact], incoming: Iterable[Fact]\n                 ) -> tuple[list[Fact], list[str]]:\n    """Returns (merged, blocked statements)."""\n    out = list(existing)\n    index = {f.statement: i for i, f in enumerate(out)}\n    blocked: list[str] = []\n    for fact in incoming:\n        position = index.get(fact.statement)\n        if position is None:\n            index[fact.statement] = len(out)\n            out.append(fact)\n            continue\n        revised = _revise(fact, out[position])\n        if revised is None:\n            blocked.append(fact.statement)\n        else:\n            out[position] = revised\n    return out, blocked\n\n\n@dataclass\nclass MemoryDelta:\n    changed: list[str] = field(default_factory=list)\n    blocked: list[str] = field(default_factory=list)  # re-proposed refuted claims\n    contested: list[str] = field(default_factory=list)\n\n\n@dataclass\nclass GameMemory:\n    """Everything known about ONE game. Never shared across games."""\n\n    game_id: str\n    level: int = 1\n    action_semantics: dict[ActionKey, list[Fact]] = field(default_factory=dict)\n    mechanics: list[Fact] = field(default_factory=list)\n    goal_evidence: list[Fact] = field(default_factory=list)\n    counterexamples: list[Fact] = field(default_factory=list)\n    winning_paths: list[WinningPath] = field(default_factory=list)\n    current_plan: list[ActionKey] = field(default_factory=list)\n    level_coordinates: dict[str, Any] = field(default_factory=dict)\n    goal_guesses: list[Fact] = field(default_factory=list)\n    object_identities: dict[str, Any] = field(default_factory=dict)\n    unpersisted: bool = False\n\n    def snapshot(self) -> dict[str, Any]:\n        """JSON-safe and REVERSIBLE -- action keys stay lists, not strings."""\n        return {\n            "game_id": self.game_id,\n            "level": self.level,\n            "action_semantics": [[list(k), [_fact(f) for f in v]]\n                                 for k, v in self.action_semantics.items()],\n            "mechanics": [_fact(f) for f in self.mechanics],\n            "goal_evidence": [_fact(f) for f in self.goal_evidence],\n            "counterexamples": [_fact(f) for f in self.counterexamples],\n            "winning_paths": [_path(p) for p in self.winning_paths],\n            "current_plan": [list(a) for a in self.current_plan],\n            "level_coordinates": json.loads(json.dumps(self.level_coordinates,\n                                                       default=_json_default)),\n            "goal_guesses": [_fact(f) for f in self.goal_guesses],\n            "object_identities": json.loads(json.dumps(self.object_identities,\n                                                       default=_json_default)),\n        }\n\n    @classmethod\n    def restore(cls, snapshot: dict[str, Any]) -> "GameMemory":\n        """Rebuild from a snapshot. Persistence that cannot be loaded is a log."""\n        return cls(\n            game_id=snapshot["game_id"],\n            level=snapshot["level"],\n            action_semantics={tuple(k): [_unfact(f) for f in v]\n                              for k, v in snapshot["action_semantics"]},\n            mechanics=[_unfact(f) for f in snapshot["mechanics"]],\n            goal_evidence=[_unfact(f) for f in snapshot["goal_evidence"]],\n            counterexamples=[_unfact(f) for f in snapshot["counterexamples"]],\n            winning_paths=[_unpath(p) for p in snapshot["winning_paths"]],\n            current_plan=[tuple(a) for a in snapshot["current_plan"]],\n            level_coordinates=snapshot["level_coordinates"],\n            goal_guesses=[_unfact(f) for f in snapshot["goal_guesses"]],\n            object_identities=snapshot["object_identities"],\n        )\n\n    def apply(self, update: MemoryUpdate,\n              known_transitions: set[int] | None = None) -> MemoryDelta:\n        validate(update, known_transitions)\n        delta = MemoryDelta()\n\n        for name in FACT_LISTS:\n            incoming = getattr(update, name)\n            if not incoming:\n                continue\n            merged, blocked = _merge_facts(getattr(self, name), incoming)\n            delta.blocked.extend(blocked)\n            if merged != getattr(self, name):\n                setattr(self, name, merged)\n                delta.changed.append(name)\n\n        if update.action_semantics:\n            for key, facts in update.action_semantics.items():\n                merged, blocked = _merge_facts(self.action_semantics.get(key, []), facts)\n                delta.blocked.extend(blocked)\n                if merged != self.action_semantics.get(key, []):\n                    self.action_semantics[key] = merged\n                    if "action_semantics" not in delta.changed:\n                        delta.changed.append("action_semantics")\n\n        if update.winning_paths:\n            for path in update.winning_paths:\n                if path not in self.winning_paths:\n                    self.winning_paths.append(path)\n                    if "winning_paths" not in delta.changed:\n                        delta.changed.append("winning_paths")\n\n        for name in ("current_plan", "level_coordinates", "object_identities"):\n            incoming = getattr(update, name)\n            if not incoming:\n                continue\n            setattr(self, name, incoming.copy())\n            delta.changed.append(name)\n\n        delta.contested = sorted(self.contested())\n        if delta.changed:\n            self.unpersisted = True\n        return delta\n\n    def contested(self) -> set[str]:\n        """Statements needing a discriminating probe rather than more assertion."""\n        out = {f.statement for name in FACT_LISTS for f in getattr(self, name)\n               if f.status is Status.CONTESTED}\n        out |= {f.statement for facts in self.action_semantics.values()\n                for f in facts if f.status is Status.CONTESTED}\n        # Two different CONFIRMED claims about one action is also a conflict.\n        for facts in self.action_semantics.values():\n            confirmed = [f.statement for f in facts if f.status is Status.CONFIRMED]\n            if len(confirmed) > 1:\n                out |= set(confirmed)\n        return out\n\n    def replayable(self, level: int, signature: str) -> list[WinningPath]:\n        return [p for p in self.winning_paths if p.usable_from(level, signature)]\n\n    def on_level_transition(self, new_level: int) -> None:\n        """Keep what was learned about the game; drop what was true of the level."""\n        self.level = new_level\n        self.current_plan = []\n        self.level_coordinates = {}\n        self.goal_guesses = []\n        self.object_identities = {}\n        self.unpersisted = True\n\n\ndef _fact(f: Fact) -> dict[str, Any]:\n    return {"statement": f.statement, "status": f.status.value,\n            "evidence": list(f.evidence)}\n\n\ndef _unfact(d: dict[str, Any]) -> Fact:\n    return Fact(d["statement"], Status(d["status"]), tuple(d["evidence"]))\n\n\ndef _path(p: WinningPath) -> dict[str, Any]:\n    return {"level": p.level, "start_signature": p.start_signature,\n            "actions": [list(a) for a in p.actions],\n            "expected_outcomes": list(p.expected_outcomes),\n            "evidence": list(p.evidence)}\n\n\ndef _unpath(d: dict[str, Any]) -> WinningPath:\n    return WinningPath(level=d["level"], start_signature=d["start_signature"],\n                       actions=tuple(tuple(a) for a in d["actions"]),\n                       expected_outcomes=tuple(d["expected_outcomes"]),\n                       evidence=tuple(d["evidence"]))\n\n\nclass MemoryJournal:\n    """Append-only before/after record. Durable per entry, and RESTORABLE.\n\n    An audit log that cannot be loaded is not persistence. `restore` rebuilds the\n    store from the last complete entry, so a run killed mid-game resumes with\n    what it had learned instead of starting over.\n    """\n\n    def __init__(self, path: str | Path) -> None:\n        self.path = Path(path)\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n\n    def record(self, memory: GameMemory, event: str, before: dict[str, Any],\n               delta: MemoryDelta) -> None:\n        entry = {"event": event, "game_id": memory.game_id, "level": memory.level,\n                 "changed": delta.changed, "blocked": delta.blocked,\n                 "contested": delta.contested, "before": before,\n                 "after": memory.snapshot()}\n        with self.path.open("a", encoding="utf-8") as handle:\n            handle.write(json.dumps(entry, separators=(",", ":")) + "\\n")\n            handle.flush()\n            os.fsync(handle.fileno())\n        memory.unpersisted = False\n\n    def entries(self) -> list[dict[str, Any]]:\n        if not self.path.exists():\n            return []\n        out = []\n        for line in self.path.read_text(encoding="utf-8").splitlines():\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                out.append(json.loads(line))\n            except json.JSONDecodeError:\n                continue  # a torn tail costs one entry, not the store\n        return out\n\n    def restore(self) -> GameMemory | None:\n        entries = self.entries()\n        return GameMemory.restore(entries[-1]["after"]) if entries else None\n\n\ndef commit(memory: GameMemory, update: MemoryUpdate, journal: MemoryJournal,\n           known_transitions: set[int] | None = None) -> MemoryDelta:\n    """Apply and persist, in that order, BEFORE the action is taken.\n\n    The action may complete the level, and the transition reset that follows\n    would discard an update that had not yet reached the journal.\n    """\n    before = memory.snapshot()\n    delta = memory.apply(update, known_transitions)\n    journal.record(memory, "update", before, delta)\n    return delta\n\n\ndef transition(memory: GameMemory, new_level: int, journal: MemoryJournal) -> None:\n    """Journal the reset too -- it destroys four fields and must be reconstructible."""\n    before = memory.snapshot()\n    memory.on_level_transition(new_level)\n    journal.record(memory, "level_transition", before,\n                   MemoryDelta(changed=list(VOLATILE)))\n')
(_memory_package / 'duck_memory_adapter.py').write_text('"""Opt-in structured memory for Duck\'s Python-tool agent.\n\nThe memory update is a sibling of ``code`` in the existing Python tool call.\nIt is validated and fsynced by the host *before* the sandbox can execute an\nenvironment action.  Nothing here changes Duck\'s policy when this subclass is\nnot selected.  Winning paths are stored but deliberately never auto-replayed\nin the memory ablation; replay would be a second treatment.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nfrom arc3.memory import (\n    Fact,\n    GameMemory,\n    MemoryJournal,\n    MemoryUpdate,\n    Status,\n    WinningPath,\n    commit,\n    transition,\n)\nfrom inference.agent.runtime_state import load_runtime_state\nfrom inference.agent.tool_agent import ToolAgent, _ToolDispatchResult\n\n\nFACT_FIELDS = ("mechanics", "goal_evidence", "counterexamples", "goal_guesses")\nUPDATE_FIELDS = set(FACT_FIELDS) | {\n    "action_semantics", "winning_paths", "current_plan", "level_coordinates",\n    "object_identities",\n}\n\n\ndef _action_key(raw: Any) -> tuple[int, int | None, int | None]:\n    if not isinstance(raw, list) or len(raw) != 3:\n        raise ValueError("an action key must be [id, x, y]")\n    key = tuple(raw)\n    if not isinstance(key[0], int) or isinstance(key[0], bool):\n        raise ValueError("action id must be an integer")\n    if any(value is not None and (not isinstance(value, int) or isinstance(value, bool))\n           for value in key[1:]):\n        raise ValueError("action coordinates must be integers or null")\n    return key\n\n\ndef _fact(raw: Any) -> Fact:\n    if not isinstance(raw, dict) or set(raw) - {"statement", "status", "evidence"}:\n        raise ValueError("a fact must contain only statement, status, and evidence")\n    evidence = raw.get("evidence", [])\n    if not isinstance(evidence, list):\n        raise ValueError("fact evidence must be a list of transition indices")\n    return Fact(\n        statement=raw.get("statement"),\n        status=Status(raw.get("status", "ASSUMED")),\n        evidence=tuple(evidence),\n    )\n\n\ndef parse_memory_update(raw: Any) -> MemoryUpdate:\n    """Convert the JSON tool argument to the tested memory contract."""\n    if not isinstance(raw, dict):\n        raise ValueError("memory_update must be a JSON object")\n    unknown = set(raw) - UPDATE_FIELDS\n    if unknown:\n        raise ValueError(f"unknown memory_update fields: {sorted(unknown)}")\n    values: dict[str, Any] = {}\n    for name in FACT_FIELDS:\n        if name in raw:\n            if not isinstance(raw[name], list):\n                raise ValueError(f"{name} must be a list")\n            values[name] = [_fact(item) for item in raw[name]]\n    if "action_semantics" in raw:\n        if not isinstance(raw["action_semantics"], list):\n            raise ValueError("action_semantics must be a list")\n        semantics: dict[tuple[int, int | None, int | None], list[Fact]] = {}\n        for item in raw["action_semantics"]:\n            if not isinstance(item, dict) or set(item) != {"action", "facts"}:\n                raise ValueError("action_semantics entries need action and facts")\n            key = _action_key(item["action"])\n            if key in semantics or not isinstance(item["facts"], list):\n                raise ValueError("duplicate action or non-list facts")\n            semantics[key] = [_fact(fact) for fact in item["facts"]]\n        values["action_semantics"] = semantics\n    if "winning_paths" in raw:\n        if not isinstance(raw["winning_paths"], list):\n            raise ValueError("winning_paths must be a list")\n        paths = []\n        for item in raw["winning_paths"]:\n            if not isinstance(item, dict) or set(item) != {\n                "level", "start_signature", "actions", "expected_outcomes", "evidence"\n            }:\n                raise ValueError("winning path needs all five fields")\n            if not isinstance(item["actions"], list) or not isinstance(item["expected_outcomes"], list):\n                raise ValueError("winning path actions and outcomes must be lists")\n            if len(item["actions"]) != len(item["expected_outcomes"]):\n                raise ValueError("a winning path needs one expected outcome per action")\n            if not isinstance(item["evidence"], list):\n                raise ValueError("winning path evidence must be a list")\n            paths.append(WinningPath(\n                level=item["level"],\n                start_signature=item["start_signature"],\n                actions=tuple(_action_key(a) for a in item["actions"]),\n                expected_outcomes=tuple(item["expected_outcomes"]),\n                evidence=tuple(item["evidence"]),\n            ))\n        values["winning_paths"] = paths\n    if "current_plan" in raw:\n        if not isinstance(raw["current_plan"], list):\n            raise ValueError("current_plan must be a list")\n        values["current_plan"] = [_action_key(item) for item in raw["current_plan"]]\n    for name in ("level_coordinates", "object_identities"):\n        if name in raw:\n            values[name] = raw[name]\n    return MemoryUpdate(**values)\n\n\ndef known_transition_ids(state_path: Path) -> set[int]:\n    """Match the zero-based ``transitions`` list visible in Duck\'s sandbox."""\n    _, history = load_runtime_state(state_path)\n    return set(range(sum(bool(entry.action.strip()) for entry in history)))\n\n\nclass MemoryToolAgent(ToolAgent):\n    """Duck with explicit, per-game, crash-proof belief revision."""\n\n    def _ensure_session(self, state_path: Path) -> None:\n        super()._ensure_session(state_path)\n        journal_path = state_path.with_suffix(".memory.jsonl")\n        if getattr(self, "_memory_journal_path", None) == journal_path:\n            self._sync_level(state_path)\n            return\n        self._memory_journal_path = journal_path\n        self._memory_journal = MemoryJournal(journal_path)\n        game_id = str(getattr(self, "_gate2_game_id", state_path.stem))\n        restored = self._memory_journal.restore()\n        if restored is not None and restored.game_id != game_id:\n            raise RuntimeError("memory journal belongs to a different game")\n        frame, _ = load_runtime_state(state_path)\n        self._memory = restored or GameMemory(game_id=game_id, level=frame.level if frame else 1)\n        self._memory_commits = 0\n        self._memory_rejections = 0\n        self._memory_transitions = 0\n        self._memory_missing_updates = 0\n        self._memory_empty_updates = 0\n        self._sync_level(state_path)\n\n    def _sync_level(self, state_path: Path) -> None:\n        """Recover a transition even if a prior tool call died after the action."""\n        frame, _ = load_runtime_state(state_path)\n        if frame is not None and frame.level != self._memory.level:\n            transition(self._memory, frame.level, self._memory_journal)\n            self._memory_transitions += 1\n\n    def _tools(self, state_path: Path) -> list[dict[str, Any]]:\n        tools = super()._tools(state_path)\n        tool = tools[0]["function"]\n        tool["description"] += (\n            " Required memory_update is validated and persisted before code runs. "\n            "Use it for concise mechanics and goal evidence, not a turn journal."\n        )\n        tool["parameters"]["properties"]["memory_update"] = {\n            "type": "object",\n            "description": (\n                "Required decision: use {} when nothing new was learned; otherwise "\n                "record at most two concise causal rules or goal hypotheses. "\n                "Fields mechanics, goal_evidence, "\n                "counterexamples, goal_guesses are arrays of facts. A fact is "\n                "{statement, status: ASSUMED|CONFIRMED|REFUTED, evidence: [transition indices]}. "\n                "CONFIRMED and REFUTED require an existing evidence index."\n            ),\n        }\n        tool["parameters"]["required"].append("memory_update")\n        return tools\n\n    def _build_user_prompt(self, action_num: int, **kwargs: Any) -> str:\n        prompt = super()._build_user_prompt(action_num, **kwargs)\n        memory = self._memory\n        facts = []\n        for name in ("mechanics", "goal_evidence", "counterexamples", "goal_guesses"):\n            for fact in getattr(memory, name):\n                facts.append(f"{name}: [{fact.status.value}] {fact.statement}")\n        for action, claims in memory.action_semantics.items():\n            for fact in claims:\n                facts.append(f"action {action}: [{fact.status.value}] {fact.statement}")\n        if memory.current_plan:\n            facts.append(f"current_plan: {memory.current_plan!r}")\n        shown = []\n        shown_chars = 0\n        for fact in facts:\n            if shown_chars + len(fact) + 1 > 3500:\n                break\n            shown.append(fact)\n            shown_chars += len(fact) + 1\n        text = "\\n".join(shown)\n        if len(shown) < len(facts):\n            text += f"\\n[{len(facts) - len(shown)} additional beliefs remain in the per-game journal.]"\n        history = kwargs.get("history_entries") or []\n        latest_id = sum(bool(entry.action.strip()) for entry in history) - 1\n        return prompt + "\\n\\nStructured per-game memory (not a journal of turns):\\n" + (\n            text or "[empty]"\n        ) + (\n            "\\nEvery python tool call must include memory_update as a JSON sibling of code. "\n            "Use {} only when you have learned no new causal rule or goal hypothesis. "\n            "After an action changes the puzzle, consider whether it supports one concise "\n            "mechanic or goal claim; record at most two new claims, not a turn journal. "\n            "For an unverified hypothesis use ASSUMED without evidence, for example "\n            \'{"code":"action([\\\'RIGHT\\\'])",\'\n            \'"memory_update":{"mechanics":[{"statement":"RIGHT moves the frame",\'\n            \'"status":"ASSUMED"}]}}. \'\n            "Use CONFIRMED or REFUTED only with evidence from a transition that already "\n            "exists. Evidence indices are zero-based positions in transitions. "\n            f"The latest available transition index is {latest_id}; never cite a future index. "\n            "To refute one, repeat its exact "\n            "statement as REFUTED with the contradictory transition index. "\n            "At a new level, keep mechanics and verified goal evidence; discard the old plan and coordinates."\n        )\n\n    def _run_python_tool(self, state_path: Path, arguments: dict[str, Any]) -> _ToolDispatchResult:\n        self._ensure_session(state_path)\n        code = str(arguments.get("code", "")).rstrip()\n        if not code:\n            return _ToolDispatchResult(json.dumps({"error": "python requires non-empty code"}))\n        try:\n            compile(code, "<python_tool>", "exec")\n        except SyntaxError as exc:\n            return _ToolDispatchResult(json.dumps({"error": f"Python syntax error: {exc}"}))\n        if "memory_update" not in arguments:\n            self._memory_missing_updates += 1\n        else:\n            try:\n                update = parse_memory_update(arguments["memory_update"])\n                if any(bool(value) for value in vars(update).values()):\n                    commit(\n                        self._memory, update, self._memory_journal,\n                        known_transitions=known_transition_ids(state_path),\n                    )\n                    self._memory_commits += 1\n                else:\n                    self._memory_empty_updates += 1\n            except (TypeError, ValueError) as exc:\n                self._memory_rejections += 1\n                return _ToolDispatchResult(json.dumps({"error": f"memory_update rejected: {exc}"}))\n        result = super()._run_python_tool(state_path, arguments)\n        frame, _ = load_runtime_state(state_path)\n        if frame is not None and frame.level != self._memory.level:\n            transition(self._memory, frame.level, self._memory_journal)\n            self._memory_transitions += 1\n        return result\n')
if str(WORKING_DIR) not in _memory_sys.path:
    _memory_sys.path.insert(0, str(WORKING_DIR))
from arc3.duck_memory_adapter import MemoryToolAgent as _MemoryToolAgent
print('MEMORY_ADAPTER_IMPORTED', flush=True)


In [ ]:
# Gate 2 paired response-cap experiment. Serving is unchanged from passing Version 11.
GATE2_COMPARISON_SMOKE = True
GATE2_SMOKE_GAME_SECONDS = 60.0
GATE2_TRIAL_SECONDS = 900.0
GATE2_SMOKE_STARTUP_LIMIT_SECONDS = 900.0
GATE2_SAFETY_MARGIN_SECONDS = 4860.0
GATE2_SOFT_STOP_GRACE_SECONDS = 120.0
GATE2_CONTROL_CAP = 0
GATE2_CANDIDATE_CAP = 0
GATE2_CONCURRENCY = 1 if GATE2_COMPARISON_SMOKE else 2
GATE2_ACTION_CAP = 400
GATE2_SEEDS = (1214842320, 656940509)

if TRUE_SUBMISSION:
    raise RuntimeError("Gate 2 comparison is local-only and must never be submitted.")
if float(getattr(target, "max_runtime_s", 0.0) or 0.0) != 32400.0:
    raise RuntimeError(
        f"Expected the 32400-second notebook budget, got {target.max_runtime_s!r}."
    )
if GATE2_SAFETY_MARGIN_SECONDS < 0.15 * float(target.max_runtime_s):
    raise RuntimeError("Gate 2 safety margin is below 15% of the notebook budget.")
if GATE1_SERVER_READY_EPOCH <= NOTEBOOK_START_EPOCH:
    raise RuntimeError("Server-ready timestamp must follow notebook start.")
if GATE1_SERVER_STARTUP_SECONDS > GATE2_SMOKE_STARTUP_LIMIT_SECONDS:
    raise TimeoutError(
        f"vLLM startup exceeded {GATE2_SMOKE_STARTUP_LIMIT_SECONDS}s: "
        f"{GATE1_SERVER_STARTUP_SECONDS}s."
    )

import threading as _gate2_threading
import inference.agent.tool_agent as _gate2_tool_agent_module
from inference.agent.tool_agent import ToolAgent as _Gate2ToolAgent
from inference.framework.solver import _HarnessGameSession as _Gate2Session

_GATE2_METRICS_LOCK = _gate2_threading.Lock()
_GATE2_METRICS = {
    "instrumentation_epoch": time.time(),
    "first_request_started_epoch": None,
    "first_response_epoch": None,
    "llm_calls_total": 0,
    "finish_reason_length_total": 0,
}
_GATE2_SESSION_METRICS = {}

if not getattr(_Gate2ToolAgent, "_gate2_call_counter_installed", False):
    _GATE2_ORIGINAL_CHAT_COMPLETION = _Gate2ToolAgent._chat_completion

    def _gate2_counted_chat_completion(self, *args, **kwargs):
        is_first = False
        started_epoch = time.time()
        with _GATE2_METRICS_LOCK:
            self._gate2_llm_calls = int(getattr(self, "_gate2_llm_calls", 0)) + 1
            _GATE2_METRICS["llm_calls_total"] += 1
            if _GATE2_METRICS["first_request_started_epoch"] is None:
                _GATE2_METRICS["first_request_started_epoch"] = started_epoch
                is_first = True
        try:
            result = _GATE2_ORIGINAL_CHAT_COMPLETION(self, *args, **kwargs)
            finish_reason = str(getattr(result, "finish_reason", "") or "").lower()
            if finish_reason == "length":
                with _GATE2_METRICS_LOCK:
                    self._gate2_finish_reason_length = int(
                        getattr(self, "_gate2_finish_reason_length", 0)
                    ) + 1
                    _GATE2_METRICS["finish_reason_length_total"] += 1
            return result
        finally:
            if is_first:
                with _GATE2_METRICS_LOCK:
                    _GATE2_METRICS["first_response_epoch"] = time.time()

    _Gate2ToolAgent._chat_completion = _gate2_counted_chat_completion
    _Gate2ToolAgent._gate2_call_counter_installed = True

if not getattr(_Gate2Session, "_gate2_action_counter_installed", False):
    _GATE2_ORIGINAL_EXECUTE_ACTION = _Gate2Session._execute_action

    def _gate2_counted_execute_action(self, *args, **kwargs):
        payload = _GATE2_ORIGINAL_EXECUTE_ACTION(self, *args, **kwargs)
        self._gate2_executed_actions = int(getattr(self, "_gate2_executed_actions", 0)) + 1
        if not bool(payload.get("board_changed")):
            self._gate2_noop_actions = int(getattr(self, "_gate2_noop_actions", 0)) + 1
        return payload

    _Gate2Session._execute_action = _gate2_counted_execute_action
    _Gate2Session._gate2_action_counter_installed = True

if not getattr(_Gate2Session, "_gate2_session_timer_installed", False):
    _GATE2_ORIGINAL_SESSION_PLAY = _Gate2Session.play

    def _gate2_traced_session_play(self):
        active_started = time.monotonic()
        try:
            return _GATE2_ORIGINAL_SESSION_PLAY(self)
        finally:
            run = getattr(self.game, "game_run", None)
            game_id = getattr(run, "game_id", str(self.game_index))
            trial_id = str(getattr(self.analyzer, "_gate2_trial_id", "unknown"))
            row = {
                "session_started": True,
                "active_wall_seconds": time.monotonic() - active_started,
                "llm_calls": int(getattr(self.analyzer, "_gate2_llm_calls", 0)),
                "finish_reason_length": int(
                    getattr(self.analyzer, "_gate2_finish_reason_length", 0)
                ),
                "no_op_actions": int(getattr(self, "_gate2_noop_actions", 0)),
                "instrumented_actions": int(getattr(self, "_gate2_executed_actions", 0)),
                "memory_commits": int(getattr(self.analyzer, "_memory_commits", 0)),
                "memory_rejections": int(getattr(self.analyzer, "_memory_rejections", 0)),
                "memory_missing_updates": int(getattr(self.analyzer, "_memory_missing_updates", 0)),
                "memory_empty_updates": int(getattr(self.analyzer, "_memory_empty_updates", 0)),
                "memory_transitions": int(getattr(self.analyzer, "_memory_transitions", 0)),
            }
            with _GATE2_METRICS_LOCK:
                _GATE2_SESSION_METRICS[(trial_id, game_id)] = row

    _Gate2Session.play = _gate2_traced_session_play
    _Gate2Session._gate2_session_timer_installed = True

print(
    "GATE2_SETTINGS "
    + json.dumps(
        {
            "smoke": GATE2_COMPARISON_SMOKE,
            "trial_seconds": (
                GATE2_SMOKE_GAME_SECONDS if GATE2_COMPARISON_SMOKE else GATE2_TRIAL_SECONDS
            ),
            "concurrency": GATE2_CONCURRENCY,
            "action_cap": GATE2_ACTION_CAP,
            "control_cap": GATE2_CONTROL_CAP,
            "candidate_cap": GATE2_CANDIDATE_CAP,
            "seeds": list(GATE2_SEEDS),
            "safety_margin_seconds": GATE2_SAFETY_MARGIN_SECONDS,
        },
        sort_keys=True,
    ),
    flush=True,
)


In [ ]:
# Gate 2 crash-proof paired comparison. Every arm shares this one vLLM session.
import asyncio as _gate2_asyncio
import copy as _gate2_copy
import os as _gate2_os
import signal as _gate2_signal


def _offline_games(env_dir: str):
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


PUBLIC_GAME_IDS = (
    "tn36-ef4dde99", "lf52-271a04aa", "cn04-2fe56bfb", "bp35-0a0ad940",
    "wa30-ee6fef47", "lp85-305b61c3", "r11l-495a7899", "tu93-0768757b",
    "sp80-589a99af", "m0r0-492f87ba", "vc33-5430563c", "ar25-0c556536",
    "ka59-38d34dbb", "sc25-635fd71a", "sk48-d8078629", "dc22-fdcac232",
    "cd82-fb555c5d", "ft09-0d8bbf25", "g50t-5849a774", "ls20-9607627b",
    "re86-8af5384d", "s5i5-18d95033", "sb26-7fbdac44", "su15-1944f8ab",
    "tr87-cd924810",
)
GATE2_GAME_IDS = ("cd82-fb555c5d", "ka59-38d34dbb")

competition_env_files = str(
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent
    / "environment_files"
)
offline_games = _offline_games(competition_env_files)
offline_by_id = {game.env_name: game for game in offline_games}
if len(offline_by_id) != len(offline_games):
    raise RuntimeError("The offline public game list contains duplicate IDs.")
missing_public = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
extra_public = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
if missing_public or extra_public:
    raise RuntimeError(
        f"Offline public game set changed; missing={missing_public}, extra={extra_public}."
    )

_gate2_expected_ids = list(GATE2_GAME_IDS[:1] if GATE2_COMPARISON_SMOKE else GATE2_GAME_IDS)
_gate2_trial_seconds = (
    GATE2_SMOKE_GAME_SECONDS if GATE2_COMPARISON_SMOKE else GATE2_TRIAL_SECONDS
)
_gate2_trial_specs = [
    {"trial_id": "W", "replicate": 0, "arm": "W", "cap": 0, "seed": GATE2_SEEDS[0]},
    {"trial_id": "M", "replicate": 0, "arm": "M", "cap": 0, "seed": GATE2_SEEDS[0]},
]
if not GATE2_COMPARISON_SMOKE:
    _gate2_trial_specs.append(
        {"trial_id": "W-repeat", "replicate": 0, "arm": "W-repeat", "cap": 0,
         "seed": GATE2_SEEDS[0]}
    )

print(
    "GATE2_PROTOCOL "
    + json.dumps(
        {
            "games": _gate2_expected_ids,
            "trial_seconds": _gate2_trial_seconds,
            "concurrency": GATE2_CONCURRENCY,
            "trials": _gate2_trial_specs,
            "note": "Budgets are matched; realized token counts are measured, not forced equal.",
        },
        sort_keys=True,
    ),
    flush=True,
)

_GATE2_ROOT = WORKING_DIR / "gate2-comparison"
_GATE2_ROOT.mkdir(parents=True, exist_ok=True)
_GATE2_PROGRESS_START = time.monotonic()
_gate2_hard_guard_triggered = False
_gate2_benchmark_error = None
_gate2_benchmark_ok = False
_gate2_teardown_error = None
_gate2_teardown_ok = False
_gate2_teardown_attempts = []
_gate2_teardown_result = {}
_gate2_post_gpu_rows = []
_gate2_completed_trials = []
_gate2_trial_benchmarks = {}


def _gate2_atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    _gate2_os.replace(tmp, path)


def _gate2_ratio(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else None


def _gate2_run_snapshot(trial_id, run, trial_started):
    session = _GATE2_SESSION_METRICS.get((trial_id, run.game_id)) or {}
    generated_tokens = sum(int(record.generated_tokens) for record in run.history)
    generated_tokens += int(getattr(run, "final_generated_tokens", 0) or 0)
    base_raw = run.base_actions_per_level
    return {
        "trial_id": trial_id,
        "game_id": run.game_id,
        "state": str(run.state),
        "final_score": float(run.final_score) if run.final_score is not None else None,
        "actions_taken": len(run.history),
        "actions_per_level": [int(value) for value in run.actions_per_level],
        "base_actions_per_level": (
            None if base_raw is None else [int(value) for value in base_raw]
        ),
        "levels_completed": int(run.levels_completed),
        "number_of_levels": int(run.number_of_levels),
        "generated_tokens": generated_tokens,
        "llm_calls": int(session.get("llm_calls", 0)),
        "finish_reason_length": int(session.get("finish_reason_length", 0)),
        "no_op_actions": int(session.get("no_op_actions", 0)),
        "instrumented_actions": int(session.get("instrumented_actions", 0)),
        "memory_commits": int(session.get("memory_commits", 0)),
        "memory_rejections": int(session.get("memory_rejections", 0)),
        "memory_missing_updates": int(session.get("memory_missing_updates", 0)),
        "memory_empty_updates": int(session.get("memory_empty_updates", 0)),
        "memory_transitions": int(session.get("memory_transitions", 0)),
        "active_wall_seconds": session.get("active_wall_seconds"),
        "trial_elapsed_seconds": time.monotonic() - trial_started,
        "notebook_elapsed_seconds": time.monotonic() - _GATE2_PROGRESS_START,
    }


def _gate2_persist_run(trial_id, run, trial_started, terminal):
    row = _gate2_run_snapshot(trial_id, run, trial_started)
    row["terminal"] = bool(terminal)
    _gate2_atomic_json(_GATE2_ROOT / trial_id / "games" / f"{run.game_id}.json", row)
    return row


def _gate2_trial_rows(trial_id):
    rows = []
    for path in sorted((_GATE2_ROOT / trial_id / "games").glob("*.json")):
        try:
            rows.append(json.loads(path.read_text()))
        except Exception as exc:
            print(f"GATE2_PROGRESS_READ_ERROR path={path} error={exc!r}", flush=True)
    return rows


def _gate2_write_progress_index():
    rows = []
    for spec in _gate2_trial_specs:
        rows.extend(_gate2_trial_rows(spec["trial_id"]))
    _gate2_atomic_json(_GATE2_ROOT / "progress.json", rows)
    jsonl_tmp = _GATE2_ROOT / "progress.jsonl.tmp"
    jsonl_tmp.write_text("".join(json.dumps(row, sort_keys=True) + "\n" for row in rows))
    _gate2_os.replace(jsonl_tmp, _GATE2_ROOT / "progress.jsonl")
    return rows


def _gate2_flush_trial(trial_id, trial_bm, trial_started):
    for run in list(trial_bm.game_runs):
        terminal = str(run.state) != "playing"
        _gate2_persist_run(trial_id, run, trial_started, terminal=terminal)
    rows = _gate2_write_progress_index()
    try:
        trial_bm._save_json()
    except Exception as exc:
        print(
            f"GATE2_BENCHMARK_SAVE_ERROR trial={trial_id} error={exc!r}",
            flush=True,
        )
    return rows


async def _gate2_progress_loop(trial_id, trial_bm, trial_started, stop_event):
    emitted = set()
    try:
        while not stop_event.is_set():
            for run in list(trial_bm.game_runs):
                if run.game_id in emitted or str(run.state) == "playing":
                    continue
                row = _gate2_persist_run(trial_id, run, trial_started, terminal=True)
                _gate2_write_progress_index()
                try:
                    trial_bm._save_json()
                except Exception as exc:
                    print(
                        f"GATE2_INCREMENTAL_SAVE_ERROR trial={trial_id} error={exc!r}",
                        flush=True,
                    )
                emitted.add(run.game_id)
                print(
                    "GATE2_GAME_COMPLETE "
                    + json.dumps(
                        {
                            "trial_id": trial_id,
                            "game_id": row["game_id"],
                            "state": row["state"],
                            "levels_completed": row["levels_completed"],
                            "actions_taken": row["actions_taken"],
                            "generated_tokens": row["generated_tokens"],
                            "trial_elapsed_seconds": row["trial_elapsed_seconds"],
                        },
                        sort_keys=True,
                    ),
                    flush=True,
                )
            await _gate2_asyncio.sleep(0.25)
    finally:
        for run in list(trial_bm.game_runs):
            if run.game_id not in emitted and str(run.state) != "playing":
                row = _gate2_persist_run(trial_id, run, trial_started, terminal=True)
                print(
                    "GATE2_GAME_COMPLETE "
                    + json.dumps(
                        {
                            "trial_id": trial_id,
                            "game_id": row["game_id"],
                            "state": row["state"],
                            "levels_completed": row["levels_completed"],
                            "actions_taken": row["actions_taken"],
                            "generated_tokens": row["generated_tokens"],
                            "trial_elapsed_seconds": row["trial_elapsed_seconds"],
                        },
                        sort_keys=True,
                    ),
                    flush=True,
                )
        _gate2_write_progress_index()


def _gate2_make_analyzer_factory(spec, solver):
    trial_id = spec["trial_id"]

    def factory(game, index):
        analyzer_cls = _MemoryToolAgent if spec['arm'] == 'M' else _Gate2ToolAgent
        analyzer = analyzer_cls(
            model=solver.model,
            timeout=solver.analyzer_timeout,
            save_request_logs=solver.save_request_logs,
        )
        analyzer._gate2_trial_id = trial_id
        analyzer._gate2_game_id = getattr(game, "env_name", str(index))
        return analyzer

    return factory


def _gate2_summarize_trial(spec, rows, gpu_seconds):
    actions = sum(int(row["actions_taken"]) for row in rows)
    calls = sum(int(row["llm_calls"]) for row in rows)
    tokens = sum(int(row["generated_tokens"]) for row in rows)
    levels = sum(int(row["levels_completed"]) for row in rows)
    no_ops = sum(int(row["no_op_actions"]) for row in rows)
    length_count = sum(int(row["finish_reason_length"]) for row in rows)
    score = sum(float(row["final_score"] or 0.0) for row in rows) / len(_gate2_expected_ids)
    return {
        **spec,
        "game_count": len(rows),
        "gpu_seconds": gpu_seconds,
        "weighted_rhae": score,
        "weighted_rhae_per_gpu_second": _gate2_ratio(score, gpu_seconds),
        "generated_tokens": tokens,
        "llm_calls": calls,
        "actions": actions,
        "completed_levels": levels,
        "games_with_progress": sum(int(row["levels_completed"] > 0) for row in rows),
        "games_won": sum(int(row["state"] == "won") for row in rows),
        "tokens_per_call": _gate2_ratio(tokens, calls),
        "actions_per_call": _gate2_ratio(actions, calls),
        "tokens_per_action": _gate2_ratio(tokens, actions),
        "tokens_per_completed_level": _gate2_ratio(tokens, levels),
        "no_op_actions": no_ops,
        "no_op_rate": _gate2_ratio(no_ops, actions),
        "finish_reason_length_count": length_count,
        "memory_commits": sum(int(row["memory_commits"]) for row in rows),
        "memory_rejections": sum(int(row["memory_rejections"]) for row in rows),
        "memory_missing_updates": sum(int(row["memory_missing_updates"]) for row in rows),
        "memory_empty_updates": sum(int(row["memory_empty_updates"]) for row in rows),
        "memory_transitions": sum(int(row["memory_transitions"]) for row in rows),
        "finish_reason_length_rate": _gate2_ratio(length_count, calls),
    }


def _gate2_proc_start_ticks(pid):
    try:
        raw = Path(f"/proc/{pid}/stat").read_text()
        close = raw.rfind(")")
        fields = raw[close + 2 :].split()
        return int(fields[19])
    except Exception:
        return None


def _gate2_gpu_rows():
    try:
        completed = subprocess.run(
            [
                "nvidia-smi",
                "--query-compute-apps=pid,process_name,used_memory,gpu_uuid",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=False,
            timeout=3.0,
        )
    except Exception as exc:
        return [{"query_error": repr(exc)}]
    if completed.returncode != 0:
        return [{"query_error": completed.stderr[-2000:], "returncode": completed.returncode}]
    result = []
    for line in completed.stdout.splitlines():
        parts = [part.strip() for part in line.split(",", 3)]
        if len(parts) == 4 and parts[0].isdigit():
            result.append(
                {
                    "pid": int(parts[0]),
                    "process_name": parts[1],
                    "used_memory_mib": parts[2],
                    "gpu_uuid": parts[3],
                    "start_ticks": _gate2_proc_start_ticks(int(parts[0])),
                }
            )
    return result


def _gate2_kill_exact(rows):
    killed = []
    for row in rows:
        pid = int(row.get("pid", -1))
        saved_ticks = row.get("start_ticks")
        current_ticks = _gate2_proc_start_ticks(pid)
        if pid <= 1 or saved_ticks is None or current_ticks != int(saved_ticks):
            continue
        try:
            _gate2_os.kill(pid, _gate2_signal.SIGKILL)
            killed.append({"pid": pid, "start_ticks": current_ticks})
        except ProcessLookupError:
            pass
    return killed


def _gate2_teardown_once(label):
    attempt = {"label": label, "commands": []}
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command ({label}): {command}", flush=True)
        try:
            completed = subprocess.run(
                command,
                shell=True,
                check=False,
                cwd=WORKING_DIR,
                env=_command_env(),
                timeout=90.0,
            )
            attempt["commands"].append(
                {"command": command, "returncode": int(completed.returncode)}
            )
        except Exception as exc:
            attempt["commands"].append({"command": command, "error": repr(exc)})
    result_path = WORKING_DIR / "vllm-server-teardown.json"
    if result_path.is_file():
        try:
            attempt["result"] = json.loads(result_path.read_text())
        except Exception as exc:
            attempt["result_error"] = repr(exc)
    else:
        attempt["result_error"] = "result artifact missing"
    _gate2_teardown_attempts.append(attempt)
    return attempt


def _gate2_terminal_gate_after_drain(
    evidence_result: dict,
    process_result: dict,
    post_gpu_rows: list[dict],
) -> tuple[bool, dict]:
    """Reconcile preserved teardown evidence with the current process table.

    The first teardown is the only attempt that can capture live vLLM metrics.
    A GPU process may still be visible briefly after that capture.  Once the
    exact owned process has drained, running teardown again cannot recapture the
    now-closed endpoint and therefore must not replace the first attempt's
    evidence.  This helper keeps the two facts separate.
    """

    owned_identity = {
        int(row["pid"]): row.get("start_ticks")
        for row in (evidence_result.get("vllm_gpu_rows_after") or [])
        if "pid" in row
    }

    def is_owned(row: dict) -> bool:
        pid = int(row.get("pid", -1))
        if pid not in owned_identity:
            return False
        expected_ticks = owned_identity[pid]
        actual_ticks = row.get("start_ticks")
        return expected_ticks is None or actual_ticks is None or actual_ticks == expected_ticks

    post_owned = [row for row in post_gpu_rows if is_owned(row)]
    final_scan = process_result.get("process_scan_final_gate") or {}
    conflict = bool(final_scan.get("root_conflict") or final_scan.get("saved_conflicts"))
    cpu_survivors = bool(
        final_scan.get("authorized_records")
        or final_scan.get("suspect_records")
        or process_result.get("full_proc_marker_survivors")
        or process_result.get("cpu_only_vllm_ple_marker_survivors")
    )
    gpu_query_error = any("query_error" in row for row in post_gpu_rows)
    checks = {
        "identity_valid": bool(evidence_result.get("identity_valid")),
        "no_process_identity_conflict": not conflict,
        "port_closed": bool(process_result.get("port_closed")),
        "no_cpu_survivors": not cpu_survivors,
        "gpu_query_succeeded": not gpu_query_error,
        "no_owned_gpu_survivors": not post_owned,
        "final_metrics_preserved": bool(evidence_result.get("final_metrics_preserved")),
        "required_artifacts_preserved": bool(
            evidence_result.get("required_artifacts_preserved")
        ),
    }
    details = {
        "checks": checks,
        "owned_pids": sorted(owned_identity),
        "post_owned": post_owned,
        "post_gpu_rows": post_gpu_rows,
    }
    return all(checks.values()), details


def _gate2_teardown_with_recovery():
    first = _gate2_teardown_once("initial")
    first_result = first.get("result") or {}
    first_codes_ok = all(item.get("returncode") == 0 for item in first["commands"])
    if first_codes_ok and first_result.get("shutdown_ok") is True:
        return True, first_result, _gate2_gpu_rows()

    owned_rows = list(first_result.get("vllm_gpu_rows_after") or [])
    owned_identity = {
        int(row["pid"]): row.get("start_ticks")
        for row in owned_rows
        if "pid" in row
    }
    fallback_kills = _gate2_kill_exact(owned_rows)
    drain_started = time.monotonic()
    drain_deadline = drain_started + 45.0
    remaining = []
    while True:
        post_rows = _gate2_gpu_rows()
        remaining = [
            row
            for row in post_rows
            if row.get("pid") in owned_identity
            and (
                owned_identity[row["pid"]] is None
                or row.get("start_ticks") is None
                or row.get("start_ticks") == owned_identity[row["pid"]]
            )
        ]
        if not remaining or time.monotonic() >= drain_deadline:
            break
        retry_rows = [
            {"pid": row["pid"], "start_ticks": owned_identity.get(row["pid"])}
            for row in remaining
        ]
        fallback_kills.extend(_gate2_kill_exact(retry_rows))
        time.sleep(0.5)
    print(
        "GATE2_GPU_DRAIN "
        + json.dumps(
            {
                "owned_pids": sorted(owned_identity),
                "fallback_kills": fallback_kills,
                "wait_seconds": time.monotonic() - drain_started,
                "remaining": remaining,
            },
            sort_keys=True,
        ),
        flush=True,
    )

    process_result = first_result
    recovered, recovery_details = _gate2_terminal_gate_after_drain(
        first_result, process_result, post_rows
    )
    if not recovered:
        # A real CPU-side survivor can require one more bounded cleanup pass.
        # Preserve the first attempt's metrics/artifact evidence and use the
        # retry only as a fresh process-state observation.
        second = _gate2_teardown_once("post_gpu_drain")
        process_result = second.get("result") or {}
        post_rows = _gate2_gpu_rows()
        recovered, recovery_details = _gate2_terminal_gate_after_drain(
            first_result, process_result, post_rows
        )

    if not recovered:
        print(
            "GATE2_TEARDOWN_RECOVERY_BLOCKED "
            + json.dumps(recovery_details, sort_keys=True),
            flush=True,
        )
        return False, process_result, post_rows

    recovered_result = dict(first_result)
    recovered_result["gpu_rows_after"] = post_rows
    recovered_result["gpu_query_error_after"] = any(
        "query_error" in row for row in post_rows
    )
    recovered_result["vllm_gpu_rows_after"] = recovery_details["post_owned"]
    recovered_result["shutdown_ok"] = True
    recovered_result["terminal_gate_recovered_after_gpu_drain"] = True
    recovered_result["terminal_gate_recovery"] = recovery_details
    _gate2_atomic_json(WORKING_DIR / "vllm-server-teardown.json", recovered_result)
    print(
        "GATE2_TEARDOWN_RECOVERED "
        + json.dumps(
            {
                "owned_pids": recovery_details["owned_pids"],
                "post_owned": recovery_details["post_owned"],
                "post_gpu_rows": post_rows,
                "shutdown_ok": True,
            },
            sort_keys=True,
        ),
        flush=True,
    )
    return True, recovered_result, post_rows


def _gate2_make_trial_benchmark(spec):
    trial_bm = _gate2_copy.deepcopy(bm)
    trial_bm.label = f"gate2-{spec['trial_id']}"
    trial_bm.job_dir = _GATE2_ROOT / spec["trial_id"]
    trial_bm.games = [offline_by_id[game_id] for game_id in _gate2_expected_ids]
    trial_bm.n_passes = 1
    trial_bm.game_weights = None
    trial_bm.solver.max_runtime_s_per_game = _gate2_trial_seconds
    trial_bm.solver.analyzer_timeout = 60.0 if GATE2_COMPARISON_SMOKE else 180.0
    trial_bm.solver.concurrency = GATE2_CONCURRENCY
    trial_bm.solver.max_actions_per_game = GATE2_ACTION_CAP
    trial_bm.solver.save_request_logs = False
    trial_bm.solver.analyzer_factory = _gate2_make_analyzer_factory(spec, trial_bm.solver)
    return trial_bm


async def _gate2_run_trial(spec, global_soft_epoch, global_hard_epoch):
    global _gate2_hard_guard_triggered
    trial_id = spec["trial_id"]
    _gate2_tool_agent_module._LOCAL_ANALYZER_MAX_OUTPUT = int(spec["cap"])
    _gate2_tool_agent_module._LOCAL_ANALYZER_SEED = int(spec["seed"])
    os.environ["LOCAL_ANALYZER_MAX_OUTPUT"] = str(spec["cap"])
    os.environ["LOCAL_ANALYZER_SEED"] = str(spec["seed"])
    trial_bm = _gate2_make_trial_benchmark(spec)
    _gate2_trial_benchmarks[trial_id] = trial_bm
    trial_started = time.monotonic()
    trial_started_epoch = time.time()
    trial_soft_epoch = min(trial_started_epoch + _gate2_trial_seconds, global_soft_epoch)
    trial_hard_epoch = min(trial_soft_epoch + 60.0, global_hard_epoch)
    if trial_soft_epoch <= trial_started_epoch:
        raise TimeoutError(f"No global runtime remains before trial {trial_id}.")
    print(
        "GATE2_TRIAL_START "
        + json.dumps(
            {
                **spec,
                "games": _gate2_expected_ids,
                "soft_end_epoch": trial_soft_epoch,
                "hard_end_epoch": trial_hard_epoch,
            },
            sort_keys=True,
        ),
        flush=True,
    )
    stop_event = _gate2_asyncio.Event()
    progress_task = _gate2_asyncio.create_task(
        _gate2_progress_loop(trial_id, trial_bm, trial_started, stop_event)
    )
    error = None
    try:
        await _gate2_asyncio.wait_for(
            trial_bm.run(
                soft_end_time=datetime.fromtimestamp(trial_soft_epoch),
                runtime_environment=target,
                minimal_diagnostics=True,
            ),
            timeout=max(1.0, trial_hard_epoch - time.time()),
        )
    except _gate2_asyncio.TimeoutError:
        _gate2_hard_guard_triggered = True
        error = f"Hard runtime guard triggered in {trial_id}."
    except Exception as exc:
        error = repr(exc)
    finally:
        stop_event.set()
        await _gate2_asyncio.gather(progress_task, return_exceptions=True)
        _gate2_flush_trial(trial_id, trial_bm, trial_started)

    rows = _gate2_trial_rows(trial_id)
    gpu_seconds = time.monotonic() - trial_started
    problems = []
    row_ids = [row["game_id"] for row in rows]
    if row_ids != sorted(_gate2_expected_ids):
        problems.append(f"coverage ids={row_ids}")
    if len(rows) != len(_gate2_expected_ids):
        problems.append(f"coverage count={len(rows)}")
    if not all(bool(row.get("terminal")) for row in rows):
        problems.append("non-terminal row")
    if any(row.get("final_score") is None for row in rows):
        problems.append("missing final score")
    if sum(int(row["actions_taken"]) for row in rows) <= 0:
        problems.append("zero actions")
    if error is not None:
        problems.append(error)
    summary = _gate2_summarize_trial(spec, rows, gpu_seconds)
    summary["validation_problems"] = problems
    _gate2_atomic_json(_GATE2_ROOT / trial_id / "summary.json", summary)
    print("GATE2_TRIAL_SUMMARY " + json.dumps(summary, sort_keys=True), flush=True)
    if problems:
        raise RuntimeError(f"Gate 2 trial {trial_id} failed validation: {problems}")
    _gate2_completed_trials.append(trial_id)
    return summary


if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
import vllm_server_watchdog as vllm_watchdog

vllm_watchdog_setup = vllm_watchdog.load_setup(BUNDLE_DIR / "serving_setup.py")
vllm_watchdog.start_background(
    vllm_watchdog_setup,
    vllm_watchdog.WatchdogConfig(
        interval_seconds=15.0,
        request_timeout_seconds=5,
        failure_threshold=4,
        max_restart_attempts=2,
    ),
)

if GATE2_COMPARISON_SMOKE:
    _gate2_global_soft_epoch = (
        GATE1_SERVER_READY_EPOCH + len(_gate2_trial_specs) * _gate2_trial_seconds + 120.0
    )
    _gate2_global_hard_epoch = _gate2_global_soft_epoch + 60.0
else:
    _gate2_budget = float(target.max_runtime_s)
    _gate2_global_hard_epoch = (
        NOTEBOOK_START_EPOCH + _gate2_budget - GATE2_SAFETY_MARGIN_SECONDS
    )
    _gate2_global_soft_epoch = _gate2_global_hard_epoch - GATE2_SOFT_STOP_GRACE_SECONDS

print(
    "GATE2_DEADLINES "
    + json.dumps(
        {
            "smoke": GATE2_COMPARISON_SMOKE,
            "server_ready_epoch": GATE1_SERVER_READY_EPOCH,
            "startup_seconds": GATE1_SERVER_STARTUP_SECONDS,
            "global_soft_epoch": _gate2_global_soft_epoch,
            "global_hard_epoch": _gate2_global_hard_epoch,
        },
        sort_keys=True,
    ),
    flush=True,
)

try:
    for _gate2_spec in _gate2_trial_specs:
        _gate2_summary = await _gate2_run_trial(
            _gate2_spec,
            _gate2_global_soft_epoch,
            _gate2_global_hard_epoch,
        )
except Exception as exc:
    _gate2_benchmark_error = repr(exc)
    print(f"GATE2_BENCHMARK_EXCEPTION {_gate2_benchmark_error}", flush=True)
finally:
    _gate2_write_progress_index()

try:
    if _gate2_benchmark_error is None:
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
        _gate2_benchmark_ok = True
except Exception as exc:
    _gate2_benchmark_error = repr(exc)
    _gate2_benchmark_ok = False
    print(f"GATE2_BENCHMARK_VALIDATION_EXCEPTION {_gate2_benchmark_error}", flush=True)
finally:
    _gate2_write_progress_index()

try:
    vllm_watchdog.stop_background(timeout_seconds=15.0)
except Exception as exc:
    print(f"GATE2_WATCHDOG_STOP_ERROR {exc!r}", flush=True)

try:
    _gate2_teardown_ok, _gate2_teardown_result, _gate2_post_gpu_rows = (
        _gate2_teardown_with_recovery()
    )
    if not _gate2_teardown_ok:
        _gate2_teardown_error = "bounded terminal gate did not pass after recovery"
except Exception as exc:
    _gate2_teardown_error = repr(exc)
    _gate2_teardown_ok = False
    _gate2_post_gpu_rows = _gate2_gpu_rows()

_gate2_lifecycle = {
    "smoke_mode": GATE2_COMPARISON_SMOKE,
    "benchmark_ok": _gate2_benchmark_ok,
    "benchmark_error": _gate2_benchmark_error,
    "completed_trials": _gate2_completed_trials,
    "expected_trials": [spec["trial_id"] for spec in _gate2_trial_specs],
    "teardown_ok": _gate2_teardown_ok,
    "teardown_error": _gate2_teardown_error,
    "hard_guard_triggered": _gate2_hard_guard_triggered,
    "teardown_attempts": _gate2_teardown_attempts,
    "post_teardown_gpu_rows": _gate2_post_gpu_rows,
    "elapsed_seconds": time.monotonic() - _GATE2_PROGRESS_START,
}
_gate2_atomic_json(_GATE2_ROOT / "lifecycle.json", _gate2_lifecycle)
print(
    "GATE2_LIFECYCLE "
    + json.dumps(
        {
            "benchmark": "ok" if _gate2_benchmark_ok else "failed",
            "benchmark_error": _gate2_benchmark_error,
            "completed_trials": _gate2_completed_trials,
            "teardown": "ok" if _gate2_teardown_ok else "failed",
            "teardown_error": _gate2_teardown_error,
            "hard_guard_triggered": _gate2_hard_guard_triggered,
            "post_teardown_gpu_rows": _gate2_post_gpu_rows,
            "elapsed_seconds": _gate2_lifecycle["elapsed_seconds"],
        },
        sort_keys=True,
    ),
    flush=True,
)


In [ ]:
# Local-only result audit: the final answer is levels, not token savings.
import json as _mem_json

_mem_lifecycle = _mem_json.loads((_GATE2_ROOT / "lifecycle.json").read_text())
_mem_rows = _gate2_write_progress_index()
_mem_summaries = []
for _mem_spec in _gate2_trial_specs:
    _mem_summary_path = _GATE2_ROOT / _mem_spec["trial_id"] / "summary.json"
    if _mem_summary_path.exists():
        _mem_summaries.append(_mem_json.loads(_mem_summary_path.read_text()))
for _mem_row in _mem_rows:
    print("MEMORY_GAME " + _mem_json.dumps(_mem_row, sort_keys=True), flush=True)
for _mem_summary in _mem_summaries:
    print("MEMORY_ARM " + _mem_json.dumps(_mem_summary, sort_keys=True), flush=True)

_mem_by_trial = {spec["trial_id"]: {
    row["game_id"]: row for row in _gate2_trial_rows(spec["trial_id"])
} for spec in _gate2_trial_specs}
_mem_pairs = {}
for _mem_game in _gate2_expected_ids:
    _mem_pairs[_mem_game] = {
        arm: _mem_by_trial.get(arm, {}).get(_mem_game, {}).get("levels_completed")
        for arm in ("W", "W-repeat", "M")
    }
print("MEMORY_PAIRED_LEVELS " + _mem_json.dumps(_mem_pairs, sort_keys=True), flush=True)

_mem_expected = [spec["trial_id"] for spec in _gate2_trial_specs]
_mem_clean = bool(
    _mem_lifecycle.get("benchmark_ok")
    and _mem_lifecycle.get("teardown_ok")
    and not _mem_lifecycle.get("hard_guard_triggered")
    and _mem_lifecycle.get("completed_trials") == _mem_expected
    and len(_mem_rows) == len(_mem_expected) * len(_gate2_expected_ids)
    and all(row.get("terminal") for row in _mem_rows)
    and not _mem_lifecycle.get("post_teardown_gpu_rows")
)
print("MEMORY_SCREEN_FINAL " + _mem_json.dumps({
    "smoke": GATE2_COMPARISON_SMOKE,
    "clean": _mem_clean,
    "benchmark_ok": _mem_lifecycle.get("benchmark_ok"),
    "teardown_ok": _mem_lifecycle.get("teardown_ok"),
    "completed_trials": _mem_lifecycle.get("completed_trials"),
}, sort_keys=True), flush=True)
print("MEMORY_SMOKE_OK" if GATE2_COMPARISON_SMOKE and _mem_clean else
      "MEMORY_SCREEN_OK" if _mem_clean else "MEMORY_SCREEN_FAILED", flush=True)
